In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import os

## TASK 1

Step 1: Create a dataset with 500,000 rows using the following schema (you will reuse this dataset throughout the lab). Use np.random.seed(42) for reproducibility.

In [2]:
np.random.seed(42)

rows = 500000

cities = ["Berlin","Tokyo","New York","Paris","London","Toronto","Sydney","Dubai","Singapore","Madrid"]

df = pd.DataFrame({
    "user_id": np.arange(1, rows+1),
    "city": np.random.choice(cities, rows),
    "score": np.random.uniform(0,100,rows),
    "active": np.random.choice([True,False], rows),
    "signup_date": pd.to_datetime("today") - pd.to_timedelta(np.random.randint(0,1095,rows), unit="d"),
    "age": np.random.randint(18,80,rows),
    "sessions": np.random.randint(0,500,rows),
    "revenue": np.random.uniform(0,1000,rows)
})

df.head()

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Sydney,43.689490,False,2026-02-01 10:46:14.080706,71,14,167.891338
1,2,Paris,97.403462,True,2025-06-02 10:46:14.080706,39,399,546.244310
2,3,Dubai,66.148238,False,2025-11-08 10:46:14.080706,23,283,544.570185
3,4,London,21.993544,False,2025-05-22 10:46:14.080706,20,205,421.757634
4,5,Sydney,91.719953,True,2023-04-02 10:46:14.080706,67,113,603.913530


Step 2: Save the dataset as a Parquet file. Then use pyarrow.parquet.ParquetFile to inspect:

The number of row groups
The number of rows and columns
The schema (column names and types)
For each column in the first row group: physical type, compressed size, and any available statistics (min, max, null count)

In [3]:
table = pa.Table.from_pandas(df)
pq.write_table(table, "users.parquet")

pf = pq.ParquetFile("users.parquet")

print("Row Groups:", pf.num_row_groups)
print("Rows:", pf.metadata.num_rows)
print("Columns:", pf.metadata.num_columns)

print("\nSchema:")
print(pf.schema)

Row Groups: 1
Rows: 500000
Columns: 8

Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 age;
  optional int32 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}



Step 3: Save the same dataset as CSV. Compare file sizes. Print both sizes in KB and the compression ratio.

In [4]:
df.to_csv("users.csv", index=False)

parquet_size = os.path.getsize("users.parquet")/1024
csv_size = os.path.getsize("users.csv")/1024

print("Parquet Size (KB):", parquet_size)
print("CSV Size (KB):", csv_size)
print("Compression Ratio:", csv_size/parquet_size)

Parquet Size (KB): 12496.576171875
CSV Size (KB): 44288.353515625
Compression Ratio: 3.5440390156866406


Step 4: Write a markdown cell explaining what information the Parquet metadata provides that CSV does not, and why that matters for query performance.

Parquet stores structured metadata like the schema (data types), row groups, column chunks, and statistics (min, max, null counts). CSV only stores raw text with no type information or statistics.

This matters because Parquet can skip unnecessary columns and even entire row groups when filtering data, making queries much faster. CSV must scan the whole file every time since it has no metadata to optimize the read.

## TASK 2

Step 1: Read the full Parquet file from Task 1 and time it.

In [5]:
%time df_full = pd.read_parquet("users.parquet")

CPU times: total: 172 ms
Wall time: 72.2 ms


Step 2: Read only 2 columns from the same Parquet file and time it.

In [6]:
%time df_two = pd.read_parquet("users.parquet", columns=["city","score"])

CPU times: total: 62.5 ms
Wall time: 43.5 ms


Step 3: Read only 2 columns from the CSV file and time it (you will need to read the entire CSV and select columns after, which is what pandas does).

In [7]:
%time df_csv = pd.read_csv("users.csv")[["city","score"]]

CPU times: total: 422 ms
Wall time: 418 ms


Step 4: Write a markdown cell explaining why Parquet selective reads are faster. Connect your explanation to the column-chunk layout you observed in Task 1.

Parquet stores each column separately, so it can read only the columns you request instead of the entire file. CSV stores data row by row as plain text, so pandas must read the whole file even if you only need two columns. Because Parquet reduces how much data is read from disk, it is much faster for selective queries.

## TASK 3

Step 1: Using PyArrow, read the Parquet file with a filter (e.g., age > 50). Time the read.

In [8]:
%time table_filtered = pq.read_table("users.parquet", filters=[("age",">",50)])
df_filtered_arrow = table_filtered.to_pandas()

CPU times: total: 109 ms
Wall time: 41.7 ms


Step 2: Read the full Parquet file without a filter and apply the same filter in pandas after loading. Time this approach.

In [9]:
%time df_full = pd.read_parquet("users.parquet")
df_filtered_pandas = df_full[df_full["age"] > 50]

CPU times: total: 109 ms
Wall time: 63 ms


In [10]:
print(len(df_filtered_arrow))
print(len(df_filtered_pandas))

233565
233565


Step 3: Compare the number of rows returned and the execution times. Write a markdown cell explaining what predicate pushdown does and why it is faster.

Predicate pushdown means applying filters while reading the file instead of after loading all the data.

With PyArrow, the filter (age > 50) is pushed down to the Parquet file. Parquet stores metadata like min and max values for each row group, so PyArrow can skip row groups that do not match the condition. This reduces the amount of data read from disk.

With pandas, the entire dataset is loaded first and the filter is applied afterward. This requires more memory and I/O, which makes it slower.

Because Parquet stores useful metadata, tools like PyArrow and DuckDB can read only the needed data, making queries faster.

Step 4: Run the same filtered query using DuckDB's SQL interface directly on the Parquet file:

import duckdb

result = duckdb.sql("SELECT * FROM 'your_file.parquet' WHERE age > 50").df()

Time it and compare with both PyArrow and pandas approaches.

In [11]:
%time result = duckdb.sql("SELECT * FROM 'users.parquet' WHERE age > 50").df()

CPU times: total: 78.1 ms
Wall time: 59.3 ms


## TASK 4

Run the same five analytical queries in both pandas and DuckDB on the dataset from Task 1. For each query, record the code and the execution time.

Query 1: Count records per city.

In [12]:
%time df.groupby("city").size()

CPU times: total: 31.2 ms
Wall time: 39.8 ms


city
Berlin       50059
Dubai        49885
London       50216
Madrid       50061
New York     50113
Paris        49931
Singapore    49724
Sydney       49982
Tokyo        49970
Toronto      50059
dtype: int64

In [13]:
%time duckdb.sql("SELECT city, COUNT(*) FROM df GROUP BY city").df()

CPU times: total: 0 ns
Wall time: 9.54 ms


,city,count_star()
0,Tokyo,49970
1,Madrid,50061
2,Singapore,49724
3,Paris,49931
4,Sydney,49982
5,Berlin,50059
6,Dubai,49885
7,Toronto,50059
8,London,50216
9,New York,50113


Query 2: Average score by city, ordered by average score descending.

In [14]:
%time df.groupby("city")["score"].mean().sort_values(ascending=False)

CPU times: total: 31.2 ms
Wall time: 30.8 ms


city
Toronto      50.311448
London       50.151899
Tokyo        50.135854
Sydney       50.075511
Dubai        50.063780
Berlin       50.013372
Paris        50.009568
Singapore    49.996347
Madrid       49.994487
New York     49.879502
Name: score, dtype: float64

In [15]:
%time duckdb.sql("SELECT city, AVG(score) AS avg_score FROM df GROUP BY city ORDER BY avg_score DESC").df()

CPU times: total: 46.9 ms
Wall time: 16 ms


,city,avg_score
0,Toronto,50.311448
1,London,50.151899
2,Tokyo,50.135854
3,Sydney,50.075511
4,Dubai,50.063780
5,Berlin,50.013372
6,Paris,50.009568
7,Singapore,49.996347
8,Madrid,49.994487
9,New York,49.879502


Query 3: For each city, find the percentage of active users whose score is above 75.

In [16]:
%time result = df[df["active"]].groupby("city").apply(lambda x: (x["score"] > 75).mean() * 100)
result

CPU times: total: 62.5 ms
Wall time: 61.2 ms


<timed exec>:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


city
Berlin       24.981993
Dubai        24.984980
London       25.101361
Madrid       24.913233
New York     24.764614
Paris        25.431900
Singapore    25.054554
Sydney       25.064785
Tokyo        25.355897
Toronto      25.538499
dtype: float64

In [17]:
%time duckdb.sql("SELECT city, AVG(CASE WHEN score>75 THEN 1 ELSE 0 END)*100 as percentage FROM df WHERE active = TRUE GROUP BY city").df()

CPU times: total: 0 ns
Wall time: 12.7 ms


,city,percentage
0,Paris,25.431900
1,Madrid,24.913233
2,Tokyo,25.355897
3,Sydney,25.064785
4,Berlin,24.981993
5,Toronto,25.538499
6,London,25.101361
7,New York,24.764614
8,Singapore,25.054554
9,Dubai,24.984980


Query 4: Find the top 10 users by score for each city (window function / rank).

In [18]:
%time df.assign(rank=df.groupby("city")["score"].rank(method="first", ascending=False))\
.query("rank <= 10")

CPU times: total: 203 ms
Wall time: 204 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,rank
2438,2439,Sydney,99.995192,True,2024-03-09 10:46:14.080706,29,406,471.077354,4.0
5789,5790,Dubai,99.977347,True,2024-06-28 10:46:14.080706,73,226,827.320908,7.0
7100,7101,Paris,99.998401,False,2023-07-22 10:46:14.080706,64,456,297.776414,3.0
8135,8136,Berlin,99.977353,False,2024-06-03 10:46:14.080706,60,271,897.345129,10.0
12513,12514,Singapore,99.986537,True,2023-12-30 10:46:14.080706,62,386,584.599793,9.0
...,...,...,...,...,...,...,...,...,...
470888,470889,Toronto,99.981591,True,2023-06-25 10:46:14.080706,42,401,482.788947,10.0
474291,474292,Berlin,99.992728,True,2025-01-02 10:46:14.080706,26,320,679.675081,2.0
476522,476523,Singapore,99.986165,True,2023-05-07 10:46:14.080706,18,415,751.097013,10.0
479832,479833,Sydney,99.987232,True,2024-11-23 10:46:14.080706,60,172,146.464851,10.0


In [19]:
%time duckdb.sql("SELECT * FROM ( SELECT *, RANK () OVER (PARTITION BY city ORDER BY score DESC) as r FROM df) WHERE r <= 10").df()

CPU times: total: 578 ms
Wall time: 91.2 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,r
0,412797,Dubai,99.999282,True,2023-06-16 10:46:14.080706,76,12,471.543321,1
1,38554,Dubai,99.998694,False,2023-05-17 10:46:14.080706,33,192,211.079528,2
2,162024,Dubai,99.991092,True,2024-07-05 10:46:14.080706,76,177,791.364907,3
3,255749,Dubai,99.990132,False,2023-11-11 10:46:14.080706,35,370,787.094999,4
4,167719,Dubai,99.983077,False,2023-06-17 10:46:14.080706,50,350,703.401709,5
...,...,...,...,...,...,...,...,...,...
95,343772,Singapore,99.991486,True,2025-01-02 10:46:14.080706,25,19,646.650679,6
96,115967,Singapore,99.989740,True,2023-04-10 10:46:14.080706,60,106,742.060602,7
97,78469,Singapore,99.987544,True,2024-09-23 10:46:14.080706,40,234,63.528953,8
98,12514,Singapore,99.986537,True,2023-12-30 10:46:14.080706,62,386,584.599793,9


Query 5: Compute a running total of scores ordered by user_id, partitioned by city.

In [20]:
df_sorted = df.sort_values("user_id")

%time q5_pandas = df_sorted.assign(running_total=df_sorted.groupby("city")["score"].cumsum())

CPU times: total: 31.2 ms
Wall time: 35 ms


In [21]:
%time duckdb.sql("SELECT *, SUM(score) OVER (PARTITION BY city ORDER BY user_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total FROM df").df()

CPU times: total: 641 ms
Wall time: 147 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,running_total
0,429557,New York,76.286959,False,2025-09-25 10:46:14.080706,72,291,527.735644,2.145873e+06
1,429560,New York,44.903271,False,2024-05-25 10:46:14.080706,74,33,708.876735,2.145918e+06
2,429572,New York,9.081913,True,2023-04-05 10:46:14.080706,54,247,805.354913,2.145927e+06
3,429578,New York,13.696694,True,2024-11-14 10:46:14.080706,25,154,440.701419,2.145941e+06
4,429585,New York,31.333690,True,2026-02-19 10:46:14.080706,38,393,324.824869,2.145972e+06
...,...,...,...,...,...,...,...,...,...
499995,120876,Madrid,51.825355,False,2023-06-25 10:46:14.080706,79,368,500.593747,6.136493e+05
499996,120892,Madrid,43.782532,False,2023-04-13 10:46:14.080706,40,163,855.817811,6.136931e+05
499997,120896,Madrid,87.129801,False,2023-11-04 10:46:14.080706,26,265,876.866966,6.137802e+05
499998,120900,Madrid,62.929480,False,2024-09-21 10:46:14.080706,59,225,393.673085,6.138431e+05


Step 5: Write a markdown cell comparing:

Which tool was easier to express each query in?

Which was faster?

For which queries did the difference matter most?

DuckDB queries are often easier to write for complex analytics using SQL. Pandas works well for simple transformations but can become complex for window functions. In most cases DuckDB was faster because it optimizes execution internally.

## TASK 5

Step 1: Create a pandas DataFrame and convert it to an Arrow Table using pyarrow.Table.from_pandas().

In [22]:
arrow_table = pa.Table.from_pandas(df)
arrow_table.schema

user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1163

Step 2: Inspect the Arrow Table schema and compare it to the pandas dtypes. Note any differences.

In [23]:
pq.write_table(arrow_table, "arrow_users.parquet")

Step 3: Write the Arrow Table to Parquet using pyarrow.parquet.write_table(). Read it back into a new Arrow Table.

In [24]:
arrow_table2 = pq.read_table("arrow_users.parquet")

Step 4: Convert the Arrow Table back to a pandas DataFrame using .to_pandas(). Verify the data matches the original.

In [25]:
df2 = arrow_table2.to_pandas()

df.equals(df2)

True

Step 5: Demonstrate the pandas dtype_backend="pyarrow" option by reading the Parquet file with Arrow-backed dtypes. Print the dtypes and compare with traditional pandas dtypes.

In [26]:
df_arrow = pd.read_parquet("users.parquet", dtype_backend="pyarrow")
df_arrow.dtypes

user_id                int64[pyarrow]
city                  string[pyarrow]
score                 double[pyarrow]
active                  bool[pyarrow]
signup_date    timestamp[ns][pyarrow]
age                    int32[pyarrow]
sessions               int32[pyarrow]
revenue               double[pyarrow]
dtype: object

Step 6: Write a markdown cell explaining the role of Arrow in the modern analytics stack. How does it connect Parquet (disk) to pandas (analysis) to DuckDB (SQL)?

Arrow acts as a bridge between different analytics tools. Parquet stores columnar data on disk, Arrow represents it efficiently in memory, and tools like pandas or DuckDB can process it quickly. This shared format improves interoperability and performance.